# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the FAIR² dataset using the `mlcroissant` library, referencing all entities by their `@id` fields for complete reproducibility.

### Dataset Source
The dataset is described using a Croissant schema available at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

> **Note:** All dataset entities—record sets, fields, columns—are referenced only by their `@id` fields.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load the metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# The dataset.metadata is a Croissant Metadata Python model object.
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Identifier: {metadata.identifier}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We'll enumerate the dataset's record sets and fields, referencing all by their `@id`.

In [ ]:
# List out all record sets and fields with their @id
def overview_of_recordsets(ds):
    print("Record Sets in dataset (referenced by @id):")
    for rs in ds.metadata.record_sets:
        print(f"- {rs['@id']}: {rs.get('name', '(no name)')}")

    print("\nFields per Record Set (@id):")
    for rs in ds.metadata.record_sets:
        print(f"\nRecord Set '@id': {rs['@id']}")
        print("Fields:")
        for field in rs['fields']:
            print(f"  - {field['@id']} (column: {field.get('column', '(N/A)')}, name: {field.get('name', '(no name)')})")

overview_of_recordsets(dataset)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis, referencing the record set and fields by their `@id`.

> Replace `<record_set_id>` with the actual `@id` you'd like to inspect (see above list; typically there is at least one relevant main table).

In [ ]:
# Identify main record sets (by @id)
# We'll extract all record set @id fields
record_sets_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}

for rs_id in record_sets_ids:
    # Load records for this record set using its @id
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)

# Show columns for the first available record set
main_rs_id = record_sets_ids[0]  # Change if another is primary
print(f"Columns in DataFrame for record set '{main_rs_id}':")
print(dataframes[main_rs_id].columns.tolist())

# Display first few rows
dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing, such as filtering, normalization, and grouping, referencing fields by their `@id`.

> Update `numeric_field_id` and `group_field_id` below with values corresponding to the dataset fields you want to analyze. See the output from Section 2 (fields per record set) for available options.

In [ ]:
# Example EDA: Filter and normalize a numeric field, group by another field
# These IDs should be changed to match those present in your dataset
# For illustration, we'll try to pick fields for protein quantity and sample/reagent type
numeric_field_id = None  # e.g., '@id' of a numeric field like 'abundance' or 'MW'
group_field_id = None    # e.g., '@id' of a category field, like 'sample_id'

# Use print statements above to fill in field IDs
# For demonstration, if you know that e.g. 'cr:field:MW' or similar is the @id for mass/weight, set:
# numeric_field_id = 'cr:field:MW'
# group_field_id = 'cr:field:sample_id'

# If fields are not found or appropriate, this cell may need adjustment after exploring Section 2 outputs

if numeric_field_id and numeric_field_id in dataframes[main_rs_id].columns:
    threshold = dataframes[main_rs_id][numeric_field_id].mean()
    filtered_df = dataframes[main_rs_id][dataframes[main_rs_id][numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Group by a field
    if group_field_id and group_field_id in dataframes[main_rs_id].columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Grouped data (mean) by {group_field_id}:")
        print(grouped_df.head())
    else:
        print("[INFO] Please set a group_field_id present in the DataFrame columns.")
else:
    print("[INFO] Please set a numeric_field_id present in the DataFrame columns from section 3.")

## 5. Visualization
Visualize the distribution of a numeric field or relationship between fields. You can use Matplotlib or Seaborn for more advanced plots. Make sure to reference fields by their `@id`.

> **Example:** Plot the normalized mass (MW) distribution across samples if available.

In [ ]:
import matplotlib.pyplot as plt

# Plot histogram for numeric field (edit field @ids as needed)
if numeric_field_id and numeric_field_id in dataframes[main_rs_id].columns:
    plt.figure(figsize=(8, 4))
    dataframes[main_rs_id][numeric_field_id].hist(bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id and group_field_id in dataframes[main_rs_id].columns:
        # Boxplot grouped by a discrete field
        dataframes[main_rs_id].boxplot(column=numeric_field_id, by=group_field_id, grid=False)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.suptitle("")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("[INFO] To plot, set 'numeric_field_id' and (optionally) 'group_field_id' to columns present in your dataframe.")

## 6. Conclusion
This notebook has demonstrated how to:
- Load FAIR² Croissant-based metadata and records using `mlcroissant`
- Systematically reference all dataset entities by their `@id`
- Explore available record sets and field structures
- Extract, filter, normalize, and visualize the dataset

> You can adapt and expand analysis by changing record set and field `@id`s as revealed in Section 2.